# Brazil landscape-efficiency frontier (via the duckdb-geo MCP)

Reproduces — in simplified form — the biodiversity × climate × net-economic-value trade-off of
**Polasky et al. 2026** (*Science* 392:1069) for **Brazil**, computing the **Pareto efficiency frontier**
and **optimal land-use scenario maps** directly from H3-hexed catalog layers through the public
**duckdb-geo MCP** server (`https://duckdb-mcp.nrp-nautilus.io/mcp`, stateless, no auth).

**Objectives (per H3 res-5 cell):**
- **Biodiversity** — max of normalized IUCN species richness / threatened richness / range-weighted richness (`iucn-richness-2025`).
- **Climate** — irrecoverable carbon, Mg C (`irrecoverable-carbon`).
- **Net economic value** — production revenue (USD/ha × cell area) minus the real land-use transition cost (`nci-frontiers`).

**Optimization** — the paper's method is a *separable* weighted-sum: with no cross-cell budget, each cell
independently picks the alternative maximizing `w·objective` (an `arg_max`); sweeping the weights traces the frontier.

### ⚠️ Scope / fidelity (read before citing numbers)
This is a **2-state, multi-production** approximation, not the paper's full 13-alternative model:
- Each cell is either **kept natural** (retains its biodiversity + carbon; econ = −restoration cost) **or converted** to the **best-net-value production use** (crop current / intensified-irrigated / intensified-rainfed, grazing, or forestry, net of its transition cost).
- The **economic axis is faithful** (real revenue + transition-cost layers, multiple production alternatives). The **biodiversity/carbon-under-production** is binarized (converted → 0), because the paper's per-alternative biodiversity (PREDICTS land-use response) and carbon-by-land-use-by-ecoregion models are **not yet ingested** (see the final section).

In [ ]:
# Stateless MCP client (no SDK needed) + a markdown-table parser.
import requests, json, io
import pandas as pd

MCP = "https://duckdb-mcp.nrp-nautilus.io/mcp"

def mcp_query(sql: str, timeout=300) -> pd.DataFrame:
    """Call the duckdb-geo MCP `query` tool and return the result as a DataFrame."""
    r = requests.post(MCP, timeout=timeout,
        headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream"},
        json={"jsonrpc":"2.0","id":1,"method":"tools/call",
              "params":{"name":"query","arguments":{"sql_query":sql}}})
    r.raise_for_status()
    payload = None
    for line in r.text.splitlines():            # SSE: one 'data:' line carries the JSON-RPC result
        if line.startswith("data:"):
            payload = json.loads(line[5:].strip())
    res = payload["result"]
    md = res.get("structuredContent",{}).get("result") or res["content"][0]["text"]
    return parse_md_table(md)

def parse_md_table(md: str) -> pd.DataFrame:
    rows = [l for l in md.splitlines() if l.strip().startswith("|")]
    if not rows: return pd.DataFrame()
    cells = lambda l: [c.strip() for c in l.strip().strip("|").split("|")]
    header = cells(rows[0]); data = [cells(l) for l in rows[2:]]
    df = pd.DataFrame(data, columns=header)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="ignore")
    return df

print(mcp_query("SELECT 42 AS hello"))

## 1. The efficiency frontier
Sweep objective weights; for each, the per-cell argmax picks natural vs best production. Report each objective as % of its Brazil-wide achievable maximum.

In [ ]:
FRONTIER_SQL = r'''
WITH bz AS (SELECT DISTINCT h3_cell_to_parent(h8,5) h5, h3_cell_to_parent(h8,0) h0
  FROM read_parquet('s3://public-overturemaps/2026-02-18.0/countries/hex/h0=*/data_0.parquet') WHERE country='BR'),
bzc AS (SELECT DISTINCT h5 FROM bz), bzh0 AS (SELECT DISTINCT h0 FROM bz),
bio AS (SELECT s.h5, s.combined_sr sr, COALESCE(t.combined_thr_sr,0) thr, COALESCE(r.combined_rwr,0) rwr
  FROM read_parquet('s3://public-iucn/hex/combined_sr/h0=*/data_0.parquet') s JOIN bzc USING(h5)
  LEFT JOIN read_parquet('s3://public-iucn/hex/combined_thr_sr/h0=*/data_0.parquet') t USING(h5)
  LEFT JOIN read_parquet('s3://public-iucn/hex/combined_rwr/h0=*/data_0.parquet') r USING(h5)),
mm AS (SELECT MIN(sr)a0,MAX(sr)a1,MIN(thr)b0,MAX(thr)b1,MIN(rwr)c0,MAX(rwr)c1 FROM bio),
bioc AS (SELECT h5, greatest((sr-a0)/(a1-a0),(thr-b0)/NULLIF(b1-b0,0),(rwr-c0)/NULLIF(c1-c0,0)) bio FROM bio,mm),
carb AS (SELECT h5, SUM(carbon) carbon FROM read_parquet('s3://public-carbon/irrecoverable-carbon-2024/hex/h0=*/data_0.parquet')
         WHERE h0 IN (SELECT h0 FROM bzh0) GROUP BY h5),
cc AS (SELECT h5, crop_current_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-current/hex/h0=*/data_0.parquet')),
ci AS (SELECT h5, crop_irrig_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-irrigated/hex/h0=*/data_0.parquet')),
cr AS (SELECT h5, crop_rainfed_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-rainfed/hex/h0=*/data_0.parquet')),
gz AS (SELECT h5, grazing_usd_ha v FROM read_parquet('s3://public-nci-frontiers/grazing-return/hex/h0=*/data_0.parquet')),
fo AS (SELECT h5, AVG(forestry_usd_ha) v FROM read_parquet('s3://public-nci-frontiers/forestry-return/hex/h0=*/data_0.parquet') GROUP BY h5),
tci AS (SELECT h5, SUM(tcost_ext_intens_irrig_usd) c FROM read_parquet('s3://public-nci-frontiers/tran-cost-extensification-intensified-irrigated/hex/h0=*/data_0.parquet') GROUP BY h5),
tcrf AS (SELECT h5, SUM(tcost_ext_intens_rainfed_usd) c FROM read_parquet('s3://public-nci-frontiers/tran-cost-extensification-intensified-rainfed/hex/h0=*/data_0.parquet') GROUP BY h5),
tcg AS (SELECT h5, SUM(tcost_grazing_expansion_usd) c FROM read_parquet('s3://public-nci-frontiers/tran-cost-grazing-expansion/hex/h0=*/data_0.parquet') GROUP BY h5),
cell AS (SELECT b.h5, b.bio, COALESCE(carb.carbon,0) carbon,
   greatest(COALESCE(cc.v,0)*25290, COALESCE(ci.v,0)*25290-COALESCE(tci.c,0), COALESCE(cr.v,0)*25290-COALESCE(tcrf.c,0),
            COALESCE(gz.v,0)*25290-COALESCE(tcg.c,0), COALESCE(fo.v,0)*25290, 0) econ_convert
  FROM bioc b LEFT JOIN carb USING(h5) LEFT JOIN cc USING(h5) LEFT JOIN ci USING(h5) LEFT JOIN cr USING(h5)
   LEFT JOIN gz USING(h5) LEFT JOIN fo USING(h5) LEFT JOIN tci USING(h5) LEFT JOIN tcrf USING(h5) LEFT JOIN tcg USING(h5)),
n AS (SELECT bio, carbon/NULLIF((SELECT MAX(carbon) FROM cell),0) carbn,
       econ_convert/NULLIF((SELECT MAX(econ_convert) FROM cell),0) econn FROM cell),
w(tag,wb,wc,we) AS (VALUES ('all-econ',0,0,1.0),('econ-lean',.1,.1,.8),('balanced',.34,.33,.33),('consv',.5,.5,0),('all-bio',1.0,0,0))
SELECT tag,
  ROUND(100.0*AVG((wb*bio+wc*carbn>=we*econn)::INT),1) pct_natural,
  ROUND(100.0*SUM(CASE WHEN wb*bio+wc*carbn>=we*econn THEN bio ELSE 0 END)/SUM(bio),1) biodiv_pct,
  ROUND(100.0*SUM(CASE WHEN wb*bio+wc*carbn>=we*econn THEN carbn ELSE 0 END)/SUM(carbn),1) carbon_pct,
  ROUND(100.0*SUM(CASE WHEN wb*bio+wc*carbn<we*econn THEN econn ELSE 0 END)/SUM(econn),1) econ_pct
FROM n CROSS JOIN w GROUP BY tag,wb,wc,we ORDER BY econ_pct'''

front = mcp_query(FRONTIER_SQL)
front

In [ ]:
import matplotlib.pyplot as plt
f = front.sort_values('econ_pct')
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(f['econ_pct'], f['biodiv_pct'], '-o', label='biodiversity retained')
ax.plot(f['econ_pct'], f['carbon_pct'], '-s', label='carbon retained')
for _,r in f.iterrows():
    ax.annotate(r['tag'], (r['econ_pct'], r['biodiv_pct']), fontsize=8, xytext=(4,4), textcoords='offset points')
ax.set_xlabel('Economic value captured (% of Brazil max)')
ax.set_ylabel('Conservation retained (% of max)')
ax.set_title('Brazil efficiency frontier (biodiversity/carbon vs economy)')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 2. Optimal land-use scenario maps
Per-cell optimal land use under a chosen weighting (natural / crop / grazing / forestry), aggregated to **H3 res 4** for display (modal scenario per cell) so the result fits the MCP table response. The full-resolution (res 5) assignment is available via `register_hex_tiles` / the geo-agent app.

In [ ]:
SCENARIO_SQL = r'''
WITH bz AS (SELECT DISTINCT h3_cell_to_parent(h8,5) h5, h3_cell_to_parent(h8,0) h0
  FROM read_parquet('s3://public-overturemaps/2026-02-18.0/countries/hex/h0=*/data_0.parquet') WHERE country='BR'),
bzc AS (SELECT DISTINCT h5 FROM bz), bzh0 AS (SELECT DISTINCT h0 FROM bz),
bio AS (SELECT s.h5, s.combined_sr sr, COALESCE(t.combined_thr_sr,0) thr, COALESCE(r.combined_rwr,0) rwr
  FROM read_parquet('s3://public-iucn/hex/combined_sr/h0=*/data_0.parquet') s JOIN bzc USING(h5)
  LEFT JOIN read_parquet('s3://public-iucn/hex/combined_thr_sr/h0=*/data_0.parquet') t USING(h5)
  LEFT JOIN read_parquet('s3://public-iucn/hex/combined_rwr/h0=*/data_0.parquet') r USING(h5)),
mm AS (SELECT MIN(sr)a0,MAX(sr)a1,MIN(thr)b0,MAX(thr)b1,MIN(rwr)c0,MAX(rwr)c1 FROM bio),
bioc AS (SELECT h5, greatest((sr-a0)/(a1-a0),(thr-b0)/NULLIF(b1-b0,0),(rwr-c0)/NULLIF(c1-c0,0)) bio FROM bio,mm),
carb AS (SELECT h5, SUM(carbon) carbon FROM read_parquet('s3://public-carbon/irrecoverable-carbon-2024/hex/h0=*/data_0.parquet') WHERE h0 IN (SELECT h0 FROM bzh0) GROUP BY h5),
cc AS (SELECT h5, crop_current_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-current/hex/h0=*/data_0.parquet')),
ci AS (SELECT h5, crop_irrig_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-irrigated/hex/h0=*/data_0.parquet')),
cr AS (SELECT h5, crop_rainfed_usd_ha v FROM read_parquet('s3://public-nci-frontiers/crop-rainfed/hex/h0=*/data_0.parquet')),
gz AS (SELECT h5, grazing_usd_ha v FROM read_parquet('s3://public-nci-frontiers/grazing-return/hex/h0=*/data_0.parquet')),
fo AS (SELECT h5, AVG(forestry_usd_ha) v FROM read_parquet('s3://public-nci-frontiers/forestry-return/hex/h0=*/data_0.parquet') GROUP BY h5),
cell AS (SELECT b.h5, b.bio, COALESCE(carb.carbon,0) carbon,
   COALESCE(cc.v,0)*25290 e_cur, COALESCE(ci.v,0)*25290 e_irr, COALESCE(cr.v,0)*25290 e_rain,
   COALESCE(gz.v,0)*25290 e_graze, COALESCE(fo.v,0)*25290 e_for
  FROM bioc b LEFT JOIN carb USING(h5) LEFT JOIN cc USING(h5) LEFT JOIN ci USING(h5) LEFT JOIN cr USING(h5) LEFT JOIN gz USING(h5) LEFT JOIN fo USING(h5)),
mx AS (SELECT MAX(carbon) cmax, MAX(greatest(e_cur,e_irr,e_rain,e_graze,e_for,0)) emax FROM cell),
lab AS (SELECT h5, bio, carbon/NULLIF(cmax,0) carbn,
   greatest(e_cur,e_irr,e_rain,e_graze,e_for,0)/NULLIF(emax,0) econn,
   CASE greatest(e_cur,e_irr,e_rain,e_graze,e_for,0)
     WHEN e_graze THEN 'grazing' WHEN e_for THEN 'forestry' WHEN e_irr THEN 'crop-irrigated'
     WHEN e_rain THEN 'crop-rainfed' WHEN e_cur THEN 'crop-current' ELSE 'none' END best_prod
  FROM cell, mx),
scen AS (SELECT h5, CASE WHEN {wb}*bio+{wc}*carbn >= {we}*econn THEN 'natural' ELSE best_prod END scenario FROM lab)
SELECT h3_cell_to_parent(h5,4) AS h4, mode(scenario) AS scenario FROM scen GROUP BY 1'''

scen = mcp_query(SCENARIO_SQL.format(wb=.1, wc=.1, we=.8))   # econ-lean weighting
print(len(scen), 'res-4 cells'); scen['scenario'].value_counts()

In [ ]:
# pip install h3 if needed
try:
    import h3
except ImportError:
    import sys, subprocess; subprocess.run([sys.executable,'-m','pip','install','-q','h3']); import h3
from matplotlib.patches import Polygon as MplPoly
from matplotlib.collections import PatchCollection

COLORS = {'natural':'#1b7837','forestry':'#5aae61','grazing':'#d9a441',
          'crop-current':'#fdae61','crop-irrigated':'#d73027','crop-rainfed':'#f46d43','none':'#cccccc'}

def cell_boundary_lonlat(cell_int):
    s = h3.int_to_str(cell_int) if hasattr(h3,'int_to_str') else format(int(cell_int),'x')
    b = h3.cell_to_boundary(s)            # [(lat,lng),...]
    return [(lng,lat) for lat,lng in b]

def plot_scenarios(df, title):
    fig, ax = plt.subplots(figsize=(7,7))
    patches, colors = [], []
    for _,r in df.iterrows():
        try: patches.append(MplPoly(cell_boundary_lonlat(int(r['h4'])), closed=True))
        except Exception: continue
        colors.append(COLORS.get(r['scenario'],'#cccccc'))
    pc = PatchCollection(patches, facecolor=colors, edgecolor='none')
    ax.add_collection(pc); ax.autoscale_view()
    ax.set_xlim(-75,-33); ax.set_ylim(-34,6); ax.set_aspect('equal')
    ax.set_title(title); ax.set_xlabel('lon'); ax.set_ylabel('lat')
    handles=[plt.Line2D([0],[0],marker='s',ls='',mfc=c,mec='none',label=k) for k,c in COLORS.items() if k!='none']
    ax.legend(handles=handles, loc='lower left', fontsize=8); plt.tight_layout(); plt.show()

plot_scenarios(scen, 'Brazil optimal land use — econ-lean weighting (.1 bio / .1 carbon / .8 econ)')

In [ ]:
# Same map at a balanced weighting — watch the conservation footprint expand
scen_bal = mcp_query(SCENARIO_SQL.format(wb=.34, wc=.33, we=.33))
plot_scenarios(scen_bal, 'Brazil optimal land use — balanced weighting (.34 / .33 / .33)')

## 3. The full 13 land-use alternatives — what's pending

This notebook uses a **2-state (natural vs best-production)** model. The paper scores **13 alternatives** per
parcel (restoration, forestry, grazing, + 10 crop options: current/expanded × irrigated/rainfed × ±BMP),
each with its **own** biodiversity, carbon, and net-value score. To reach the faithful 13-alternative frontier:

| Needed | Status |
|---|---|
| **Per-alternative economic value** (revenue − transition cost) | ✅ mostly: revenue layers + **5 of 14** transition costs hexed. **Pending: 9 transition-cost layers** (BMP / fixed-area / current-practices / forestry-expansion — raw tifs are in the CC0 deposit). |
| **Per-alternative carbon** (carbon by land-use × ecoregion) | ❌ **pending**: the deposit's `carbon_table_2022-08-10.csv` + `new_ecoregions.tif` aren't ingested. We have *current* irrecoverable carbon only. |
| **Per-alternative biodiversity** (richness modified by land-use intensity via **PREDICTS**) | ❌ **pending**: PREDICTS lookups are in the deposit, but the InVEST biodiversity-under-scenario model isn't reproduced. We have *current* richness only. |
| **Allowed-alternative masks** (crop suitability, slope, sustainable-irrigation) | ❌ **pending**: masks are in the deposit, not hexed — needed to forbid alternatives per cell. |
| **Faithful current land-use assignment** (grazing vs natural; managed vs natural forest) | 🔄 in progress: ESA land cover hexed (`esa-lc-2025`); **Lesiv 2022 forest management** (CC-BY) downloading for the forestry split; grazing needs an open pasture layer (the paper's Hoskins is no-distribution-licensed). |

**Two routes to the full result:** (a) extend the H3 ingest (the 9 transition costs + carbon-by-LU lookup + a
biodiversity-per-LU model) and run this same separable argmax over 13 alternatives; or (b) run the authors'
released Julia/Python code (Dryad `10.5061/dryad.9s4mw6mxn`) on the CC0 input deposit for the exact frontier.
The economic axis and the optimization machinery shown here are already faithful; the gap is the
per-alternative biodiversity/carbon scoring.